In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv

In [2]:
# load the environment variable
load_dotenv()

True

In [3]:
# Define the Hugging Face endpoint
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    # repo_id="openai/gpt-oss-120b",
    task="text-generation",
    max_new_tokens=2048,
)

# Define the model
model = ChatHuggingFace(llm=llm)

e:\Langgraph CampusX\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [10]:
def gen_outline(state: BlogState) -> BlogState:

    title = state["title"]

    prompt = f"Generate a detailed outline for a blog post with the title: {title}"

    outline = model.invoke(prompt).content

    state["outline"] = outline

    return state

In [11]:
def gen_content(state: BlogState) -> BlogState:

    title = state["title"]
    outline = state["outline"]

    prompt = f"Generate a detailed blog post based on the following title {title} and outline \n {outline}"

    content = model.invoke(prompt).content

    state["content"] = content

    return state

In [12]:
# define the graph
graph = StateGraph(BlogState)

# add nodes
graph.add_node("generate outline", gen_outline)
graph.add_node("generate content", gen_content)

# add edges
graph.add_edge(START, "generate outline")
graph.add_edge("generate outline", "generate content")
graph.add_edge("generate content", END)

# define the workflow
workflow = graph.compile()

In [13]:
initial_state = {'title': "The future of AI in education"}

final_state = workflow.invoke(initial_state)

print(final_state["outline"])

### Blog Post Outline: The Future of AI in Education

#### Introduction
- Brief introduction to AI in education
- Importance of AI in modern educational settings
- Thesis statement: AI is poised to revolutionize education by enhancing personalized learning experiences, improving accessibility, and facilitating greater efficiency in educational processes.

#### Advancements in AI for Education
1. **Customized Learning Paths**
   - Personalized learning algorithms
   - Adaptivity and pacing of learning materials
   - Case studies of successful AI-driven adaptive learning systems

2. **Automated Grading and Feedback**
   - Benefits of automated grading systems
   - Enhancements in providing timely, constructive feedback
   - Limitations and areas for improvement

3. **Virtual Assistants and Tutoring**
   - Role of AI in virtual tutoring
   - Integration of chatbots and virtual assistants in classrooms
   - Examples of AI-driven tutoring platforms

#### Enhancing Accessibility and Inclusiv

In [14]:
print(final_state["content"])

### The Future of AI in Education

#### Introduction
Artificial Intelligence (AI) has been making significant strides in various sectors, and education is no exception. AI is becoming an integral part of modern educational settings, offering new opportunities to enhance learning experiences, improve accessibility, and streamline administrative tasks. As AI technologies continue to evolve, they are poised to revolutionize the way we teach and learn. This blog post will explore the advancements in AI for education, its impact on accessibility and inclusivity, efficiency in administration, and the ethical and societal implications.

#### Advancements in AI for Education
**Customized Learning Paths**
One of the most significant ways AI is enhancing education is through personalized learning paths. Personalized learning algorithms can adapt to individual students' learning styles, paces, and preferences, providing a tailored learning experience. For instance, platforms like Knewton and Drea